# Clustering & Validation on Real Radio Data — HTRU2

The **HTRU2** survey flagged periodic radio signals. Each row is a *candidate*
that is either a real **pulsar** or **RFI/noise**. Our job: run the full
unsupervised pipeline — **scale → reduce → cluster → validate** — and see where
it succeeds and where it fails.

We *have* the labels, but we'll cluster as if we didn't, then use the labels
only at the end to check ourselves.

**Dataset**: 17,898 candidates, 8 numeric features (4 from the integrated pulse
profile, 4 from the DM–SNR curve), ≈9% real pulsars. Source: UCI ML Repository /
Lyon et al. 2016.

### Two scores we'll use throughout: *purity* and *completeness*

Once we have clusters *and* the true labels, we judge a cluster against the science
class (pulsars) with two numbers — keep both in mind, we reference them constantly:

- **Purity** (precision): of the points *inside* a cluster, what fraction are real
  pulsars? High purity → few false alarms.
- **Completeness** (recall): of *all* the pulsars in the data, what fraction landed
  in this cluster? High completeness → few missed.

They are **independent**: a big loose cluster can be very complete but impure; a
tight one can be pure but miss most pulsars. A good result needs both.

> **Important:** purity and completeness can only be computed *because we have the
> labels here*. On a genuinely unlabelled dataset — the usual case — you have **no
> way to measure them**. There you fall back on label-free heuristics like the
> **silhouette** (geometry only), knowing they score the shape of the clustering,
> not whether you actually found the science class.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples

# UMAP is optional. If it's missing, try to install it (works on Colab),
# then fall back to PCA only if that fails too.
# Note: the import name is `umap` but the pip package is `umap-learn`.
try:
    import umap
    HAS_UMAP = True
except ImportError:
    import sys, subprocess
    print("umap-learn not found - attempting to install it...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)
    try:
        import umap
        HAS_UMAP = True
    except ImportError:
        HAS_UMAP = False
        print("Install failed - the notebook will use PCA where UMAP is mentioned.")

rng = np.random.default_rng(0)


## Setup: load the data

The **HTRU2** dataset downloads automatically from the UCI archive and is cached
locally after the first run. If you've already staged the file somewhere, point
`DATA_PATH` at your copy and it'll use that instead of downloading.

`HTRU_2.csv` has 8 feature columns plus a final class column (0 = noise,
1 = pulsar) and **no header**.


In [ ]:
import os, io, zipfile, urllib.request

# Local copy if you have one; otherwise it's fetched from UCI (works on Colab).
DATA_PATH = "HTRU_2.csv"
UCI_URL = "https://archive.ics.uci.edu/static/public/372/htru2.zip"

if not os.path.exists(DATA_PATH):
    print("Downloading HTRU2 from the UCI archive...")
    blob = urllib.request.urlopen(UCI_URL, timeout=60).read()
    csv_bytes = zipfile.ZipFile(io.BytesIO(blob)).read("HTRU_2.csv")
    with open(DATA_PATH, "wb") as f:
        f.write(csv_bytes)
    print(f"Saved {DATA_PATH}")

cols = [
    "prof_mean", "prof_std", "prof_kurt", "prof_skew",
    "dm_mean", "dm_std", "dm_kurt", "dm_skew", "label",
]
# The CSV uses old-style line endings; text mode normalises them before parsing.
text = open(DATA_PATH).read().replace("\r", "\n")
raw = np.loadtxt(io.StringIO(text), delimiter=",")
X = raw[:, :8]
y = raw[:, 8].astype(int)

print(f"{X.shape[0]} candidates, {X.shape[1]} features")
print(f"pulsars: {y.sum()} ({100 * y.mean():.1f}%)  |  noise: {(y == 0).sum()}")


### Look at the features before modelling

Each candidate is 8 numbers: 4 summarise the **integrated pulse profile** (the
folded radio pulse shape) and 4 summarise the **DM–SNR curve** (how the signal
strength varies with dispersion measure). You don't need the radio astronomy — the
point is just to *see* which numbers behave differently for pulsars vs noise.

Histogram each feature, split by class (we peek at the labels here only to build
intuition):

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, (ax, name) in enumerate(zip(axes.ravel(), cols[:8])):
    ax.hist(X[y == 0, i], bins=60, density=True, alpha=0.5,
            color="#95a5a6", label="noise")
    ax.hist(X[y == 1, i], bins=60, density=True, alpha=0.5,
            color="#e74c3c", label="pulsar")
    ax.set_title(name); ax.set_yticks([])
axes[0, 0].legend()
fig.tight_layout(); plt.show()

**Think about it:** which *single* feature separates pulsars from noise most
cleanly? (Look for the histogram where the two colours overlap the least.) Keep
your answer in mind — you'll put it to work in the final exercise.

## Part 1 — Scale the features

The 8 features live on very different scales (means vs kurtoses). Distance-based
methods would let the large-scale columns dominate. Standardise first.

**Exercise:** standardise `X` so every column has mean 0 and unit variance.

In [ ]:
# YOUR CODE HERE
# Standardise X with StandardScaler so every column has mean 0 and unit variance.
# Store the result in `Xs`.
raise NotImplementedError("Standardise the features into Xs")

print("means ~0:", np.round(Xs.mean(axis=0), 3))
print("stds  ~1:", np.round(Xs.std(axis=0), 3))

## Part 2 — Reduce and look

Project to 2D so we can *see* the structure. We'll compare PCA (linear, global)
with a local method (UMAP if available, else t-SNE). Colour by the true label —
only to judge the embedding; the clustering below won't use it.

In [ ]:
pca = PCA(n_components=2)
emb_pca = pca.fit_transform(Xs)

if HAS_UMAP:
    emb_local = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Xs)
    local_name = "UMAP"
else:
    idx = rng.choice(len(Xs), 4000, replace=False)
    emb_local = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(Xs[idx])
    local_name = "t-SNE (4k subsample)"

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, emb, lab, title in [
    (axes[0], emb_pca, y, "PCA"),
    (axes[1], emb_local, (y if HAS_UMAP else y[idx]), local_name),
]:
    ax.scatter(emb[lab == 0, 0], emb[lab == 0, 1], s=6, alpha=0.3,
               color="#95a5a6", label="noise")
    ax.scatter(emb[lab == 1, 0], emb[lab == 1, 1], s=6, alpha=0.3,
               color="#e74c3c", label="pulsar")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.legend()
plt.tight_layout(); plt.show()

The pulsars occupy a distinct region — the structure is *there* for an
algorithm to find. Now let's try to recover it without the labels.

## Part 3 — Clustering with K-means

### 3a. The naive attempt: K=2

Two classes, so K=2 seems obvious. Cluster on the scaled features.

**Exercise:** fit `KMeans` with 2 clusters on `Xs`, store labels in `km_labels`.

In [ ]:
# YOUR CODE HERE
# Fit KMeans with 2 clusters (n_init=10, random_state=0) on Xs.
# Store its `.labels_` in `km_labels`.
raise NotImplementedError("Cluster Xs with K-means, K=2")

# How well does each k-means cluster line up with the real pulsar class?
for c in (0, 1):
    in_c = km_labels == c
    print(f"cluster {c}: {in_c.sum():5d} points, "
          f"{100 * y[in_c].mean():.1f}% pulsars")
# Visual cue: the K=2 split (left) vs the truth (right), on the PCA embedding.
# If K-means had found the pulsars, the left panel would match the right.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for c, color in [(0, '#2980b9'), (1, '#f39c12')]:
    m = km_labels == c
    axes[0].scatter(emb_pca[m, 0], emb_pca[m, 1], s=6, alpha=0.4,
                    color=color, label=f'cluster {c}')
axes[0].set_title('K-means clusters (K=2)'); axes[0].legend()
axes[1].scatter(emb_pca[y == 0, 0], emb_pca[y == 0, 1], s=6, alpha=0.3,
                color='#95a5a6', label='noise')
axes[1].scatter(emb_pca[y == 1, 0], emb_pca[y == 1, 1], s=10, alpha=0.7,
                color='#e74c3c', label='pulsar')
axes[1].set_title('True labels'); axes[1].legend()
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

Look at the pulsar fraction in each cluster. On HTRU2 the smaller cluster comes
out only **≈62% pulsar** (the rest is noise contamination) while still capturing
most of the real pulsars. The imbalance pulls the K=2 boundary so the rare class
never gets a clean cluster of its own. **This is the imbalance trap.**

But notice we haven't *measured* anything yet — we only spotted the problem because
we peeked at the labels. Next we validate the way you'd have to *without* them.

### 3b. Validate it — does the geometry match the science?

Two kinds of check (from the lecture): **internal** scores that use only the
geometry, and **external** scores that compare to known labels.

#### Internal: silhouette (no labels)

Silhouette scores the *geometry*: tight, well-separated clusters score near 1.
Use it to compare clusterings or choose K.

**Exercise:** compute the mean silhouette for `km_labels` over `Xs`.

In [ ]:
# silhouette on 18k points is slow — subsample
sub = rng.choice(len(Xs), 5000, replace=False)

# YOUR CODE HERE
# Compute the mean silhouette of km_labels over Xs, on the `sub` subsample.
# Store it in `sil_km`. (silhouette_score(features, labels))
raise NotImplementedError("Compute the mean silhouette")

print(f"K-means (K=2) mean silhouette: {sil_km:.3f}")

A respectable mean can still hide a ragged cluster — so plot the per-point
silhouettes, exactly as in the lecture. Wide bars near 1 = points sitting deep in
their cluster; short bars = near a boundary; bars crossing 0 = probably misassigned.

In [ ]:
# The silhouette *plot* from the lecture: one bar ("blade") per point, grouped by
# cluster and sorted within each. Read the shape, not just the mean.
# YOUR CODE HERE
# Compute the per-point silhouette values for km_labels over Xs[sub]. Store in `sil_vals`.
raise NotImplementedError("Compute per-point silhouette values")

fig, ax = plt.subplots(figsize=(8, 3))
y_lower = 0
colors = ['#2980b9', '#f39c12']
for k in (0, 1):
    vals = np.sort(sil_vals[km_labels[sub] == k])
    ax.barh(range(y_lower, y_lower + len(vals)), vals, height=1.0,
            color=colors[k], alpha=0.7, label=f'cluster {k}')
    y_lower += len(vals)
ax.axvline(sil_km, color='#e74c3c', ls='--', label=f'mean = {sil_km:.2f}')
ax.set_xlabel('silhouette value'); ax.set_yticks([]); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

#### The catch: silhouette vs. truth

A respectable silhouette can coexist with a clustering that *missed the science*.
Since we have labels here, measure how pure the clusters actually are against the
pulsar class.

**Exercise:** for the cluster with the higher pulsar fraction, compute its
*purity* (fraction that are real pulsars) and *completeness* (fraction of all
pulsars it captured).

In [ ]:
# purity = pulsars in cluster / cluster size; completeness = captured / all pulsars
# YOUR CODE HERE
# 1. For each cluster (0, 1) get its pulsar fraction; pick `best` = the higher one.
# 2. purity = pulsar fraction of that cluster.
# 3. completeness = pulsars in that cluster / total pulsars (y.sum()).
raise NotImplementedError("Compute purity and completeness for the best cluster")

print(f"best k-means cluster: purity={purity:.2f}, completeness={completeness:.2f}")
print(f"silhouette said {sil_km:.2f} — geometry looked fine, but did we find the pulsars?")

### 3c. The fix: reduce, then cluster

The silhouette looked healthy but the cluster was only ≈62% pulsar — geometry
clean, science missed. The cure is to **reduce first**: cluster the **PCA
embedding** from Part 2 instead of the raw features, where the pulsars already pull
away along the first component. That strips the noise directions dragging the K=2
boundary, so the rare class can finally get a cluster of its own.

**Exercise:** cluster the 2D PCA embedding (`emb_pca`) with K-means (K=2), then
re-check purity, completeness *and* silhouette. Does an honest internal score now
line up with actually recovering the pulsars?

In [ ]:
# YOUR CODE HERE
# Reduce first, then cluster:
#   - fit KMeans(K=2, n_init=10, random_state=0) on the PCA embedding `emb_pca`;
#     store labels in `lab_pca`.
#   - for the cluster with the higher pulsar fraction, compute `purity_pca`
#     and `completeness_pca` (same definitions as above).
#   - compute `sil_pca` = silhouette over emb_pca[sub] / lab_pca[sub].
raise NotImplementedError("Cluster the PCA embedding and re-score")

print('reduce-then-cluster (PCA 2-D, K=2):')
print(f'  purity       = {purity_pca:.2f}   (raw features: {purity:.2f})')
print(f'  completeness = {completeness_pca:.2f}')
print(f'  silhouette   = {sil_pca:.2f}   (raw features: {sil_km:.2f})')

# Visual: the cluster split (left) vs the truth (right)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for c, color in [(0, '#2980b9'), (1, '#f39c12')]:
    m = lab_pca == c
    axes[0].scatter(emb_pca[m, 0], emb_pca[m, 1], s=6, alpha=0.4,
                    color=color, label=f'cluster {c}')
axes[0].set_title('Cluster the PCA embedding (K=2)'); axes[0].legend()
axes[1].scatter(emb_pca[y == 0, 0], emb_pca[y == 0, 1], s=6, alpha=0.3,
                color='#95a5a6', label='noise')
axes[1].scatter(emb_pca[y == 1, 0], emb_pca[y == 1, 1], s=10, alpha=0.7,
                color='#e74c3c', label='pulsar')
axes[1].set_title('True labels'); axes[1].legend()
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

**What happened.** Clustering the PCA embedding instead of the raw features lifts
purity from ≈0.62 to ≈0.94 *and* nudges the silhouette up (≈0.59 → ≈0.68) — this
time the honest internal score and the science agree. The trade-off is a small drop
in completeness (≈0.78 → ≈0.70): the tighter cluster is purer but leaves a few
borderline pulsars outside. Reducing first stripped the noise directions that were
dragging the K=2 boundary, so the rare class finally gets a cluster of its own.

## Part 4 — Clustering with DBSCAN

K-means worked once we reduced — but it *had* to split everything into K convex
cells. DBSCAN takes a different route: it grows clusters through dense regions and
leaves the sparse gaps as **noise** (label `-1`). No `K` to choose, and the rare
class can fall out as its own dense island — *if* one exists.

That "if" is the catch: DBSCAN only separates clusters that are **density-separated**.
So the question is whether the pulsars form a dense island — and that depends
entirely on the representation. We'll run the same three: raw features, then each
embedding from Part 2.

In [ ]:
# --- DBSCAN attempt 1: the raw 8-D scaled features ---
# YOUR CODE HERE
# Fit DBSCAN(eps=1.2, min_samples=10) on Xs -> `db_raw`.
# Count the clusters into `n_raw` (exclude the noise label -1).
raise NotImplementedError("Run DBSCAN on the raw scaled features")

print(f"raw 8-D: {n_raw} cluster(s), "
      f"{(db_raw.labels_ == -1).sum()} points flagged as noise")


def plot_dbscan_vs_truth(emb, labels, lab_true, title):
    """Left: DBSCAN's labels (noise in grey). Right: the truth. Same 2-D view."""
    palette = ['#2980b9', '#27ae60', '#f39c12', '#8e44ad', '#16a085', '#c0392b']
    cols = ['#bdc3c7' if l == -1 else palette[l % len(palette)] for l in labels]
    ncl = len(set(labels)) - (1 if -1 in labels else 0)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].scatter(emb[:, 0], emb[:, 1], s=6, alpha=0.5, c=cols)
    axes[0].set_title(f'{title}\n{ncl} cluster(s) found, noise in grey')
    axes[1].scatter(emb[lab_true == 0, 0], emb[lab_true == 0, 1], s=6, alpha=0.3,
                    color='#95a5a6', label='noise')
    axes[1].scatter(emb[lab_true == 1, 0], emb[lab_true == 1, 1], s=10, alpha=0.7,
                    color='#e74c3c', label='pulsar')
    axes[1].set_title('True labels'); axes[1].legend()
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()


# Show DBSCAN's raw-feature labels on the familiar PCA view.
plot_dbscan_vs_truth(emb_pca, db_raw.labels_, y, 'DBSCAN on raw 8-D features')

**Failed.** Everything lands in one cluster (plus a few stray noise points) — the
left panel is a single colour where the right clearly has a red pulsar region. In
8-D the distances inflate and flatten out (every point is "far" from every other),
so the density contrast DBSCAN relies on washes out: there's no gap to cut along.

Same lesson as raw K-means in 3a — clustering the **raw** features is the problem.
So let's *reduce first*, just as we did for K-means. But which reduction? We built
two in Part 2, and here they behave very differently.

In [ ]:
# --- DBSCAN attempt 2: the PCA embedding ---
# We need an eps. Rather than guess, set it from the k-distance "knee" — the same
# idea as the lecture's k-distance plot (we'll draw that plot for the next attempt).
from sklearn.neighbors import NearestNeighbors


def knee_eps(emb, min_samples, pct=99):
    """A robust eps: a high percentile of each point's distance to its
    min_samples-th neighbour. Adapts to whatever the embedding's scale is."""
    kdist = NearestNeighbors(n_neighbors=min_samples).fit(emb).kneighbors(emb)[0][:, -1]
    return np.percentile(kdist, pct)


# YOUR CODE HERE
# Use knee_eps to pick `eps_pca` (min_samples=20) for emb_pca, then
# fit DBSCAN(eps=eps_pca, min_samples=20) on emb_pca -> `db_pca`.
raise NotImplementedError("Run DBSCAN on the PCA embedding")

plot_dbscan_vs_truth(emb_pca, db_pca.labels_, y,
                     f'DBSCAN on the PCA embedding (eps={eps_pca:.2f})')

**Still one blob.** PCA preserves *global variance*, so it lays the pulsars down as
a **connected tail** hanging off the main cloud — they trail away without a
low-density gap in between. K-means happily slices straight through that tail (which
is why reduce→K-means worked in 3c), but DBSCAN only cuts where the density actually
drops, and here it never does.

What we want is a reduction that *pulls the rare class apart* into its own island —
**local** structure, not global variance. That is exactly what UMAP optimises for.

In [ ]:
# k-distance plot from the lecture: sort every point's distance to its 20th
# neighbour. The "knee" — where the curve lifts off the flat plateau into the
# sparse tail — is a good starting eps.
emb_local_lab = y if HAS_UMAP else y[idx]   # labels matching the Part-2 embedding
ms = 20

# YOUR CODE HERE
# Build the k-distance curve for emb_local: for each point get the distance to its
# `ms`-th nearest neighbour (NearestNeighbors), then sort ascending into `kdist`.
raise NotImplementedError("Build the sorted k-distance curve")

# Locate the knee automatically: the point on the sorted curve furthest from the
# straight line joining its two ends (a simple "kneedle"). This is what your eye
# does on the lecture's k-distance plot.
xs = np.arange(len(kdist), dtype=float)
x0, x1, y0, y1 = xs[0], xs[-1], kdist[0], kdist[-1]
dist_to_chord = np.abs((y1 - y0) * xs - (x1 - x0) * kdist + x1 * y0 - y1 * x0)
knee = int(np.argmax(dist_to_chord))
eps_knee = float(kdist[knee])

plt.figure(figsize=(5.5, 3.4))
plt.plot(kdist, lw=2, color='#2980b9')
plt.axhline(eps_knee, color='#e74c3c', ls='--', label=f'knee eps ≈ {eps_knee:.2f}')
plt.scatter([knee], [eps_knee], color='#e74c3c', zorder=5)
plt.xlabel(f'points, sorted by distance to {ms}th neighbour')
plt.ylabel(f'{ms}th-neighbour distance')
plt.title(f'k-distance plot ({local_name} embedding)')
plt.legend(); plt.tight_layout(); plt.show()
print(f"knee suggests eps ≈ {eps_knee:.2f} — a starting point, not gospel")

In [ ]:
# YOUR CODE HERE
# Fit DBSCAN(eps=eps_knee, min_samples=ms) on the local embedding emb_local -> `db_local`.
raise NotImplementedError("Run DBSCAN at the knee eps on the local embedding")

plot_dbscan_vs_truth(emb_local, db_local.labels_, emb_local_lab,
                     f'DBSCAN on the {local_name} embedding (eps={eps_knee:.2f})')

# How clean is the densest pulsar cluster? (purity / completeness, as defined up top)
best, best_frac = None, -1.0
for c in set(db_local.labels_) - {-1}:
    frac = emb_local_lab[db_local.labels_ == c].mean()
    if frac > best_frac:
        best, best_frac = c, frac
in_best = db_local.labels_ == best
print(f"densest pulsar cluster: purity={emb_local_lab[in_best].mean():.2f}, "
      f"completeness={emb_local_lab[in_best].sum() / emb_local_lab.sum():.2f}")

**Success.** UMAP turns the pulsar tail into a *separated island*, and DBSCAN snaps
straight onto it — a clean cluster at ≈0.97 purity that recovers ≈0.74 of all
pulsars, leaving the fuzzy in-between candidates as noise. It found the cluster
count on its own; we never told it `K`.

The point isn't "UMAP beats PCA" — it's that **the representation decides what the
clusterer can see**. DBSCAN needs density gaps, and only the local embedding gave
it one. K-means needs convex blobs, and PCA was enough for that (Part 3c). Match the
reduction to the algorithm.

**A note on how forgiving this is.** The density gap UMAP opens up is so wide that
`eps` barely matters here — even a deliberately loose choice (e.g. the 99th
percentile of the k-distances, well *above* the knee) still isolates the same
island. That's a luxury of a clean embedding, not a general rule: on a tighter gap
(or the PCA embedding above) the knee genuinely matters. See for yourself —

In [ ]:
# Now tune it yourself: drag the slider and watch the clustering AND the scores
# move. Too small → the island shatters into noise. Too large → it merges into the
# main blob and purity collapses. Start at the knee value above.
from ipywidgets import interact, FloatSlider


def cluster_at_eps(eps):
    labels = DBSCAN(eps=eps, min_samples=ms).fit(emb_local).labels_
    best, best_frac = None, -1.0
    for c in set(labels) - {-1}:
        frac = emb_local_lab[labels == c].mean()
        if frac > best_frac:
            best, best_frac = c, frac
    n_noise = int((labels == -1).sum())
    if best is None:
        print(f"eps={eps:.2f}: no clusters — everything is noise ({n_noise} points)")
    else:
        in_best = labels == best
        purity = emb_local_lab[in_best].mean()
        completeness = emb_local_lab[in_best].sum() / emb_local_lab.sum()
        print(f"eps={eps:.2f}:  best pulsar cluster  purity={purity:.2f}  "
              f"completeness={completeness:.2f}  |  {n_noise} noise points")
    plot_dbscan_vs_truth(emb_local, labels, emb_local_lab,
                         f'DBSCAN on the {local_name} embedding (eps={eps:.2f})')


interact(cluster_at_eps,
         eps=FloatSlider(min=round(eps_knee * 0.3, 2), max=round(eps_knee * 3, 2),
                         step=round(eps_knee * 0.1, 2), value=round(eps_knee, 2)));

## Part 5 — Reflect

Two very different algorithms — K-means (centroids) and DBSCAN (density) — both
recovered the pulsars, but *only after the right reduction*: K-means needed PCA,
DBSCAN needed UMAP. On the raw features, both failed. **The representation was the
lever, not the algorithm.**

### Takeaways

- **Scaling matters**: unscaled features let the big-magnitude columns dominate distance.
- **Imbalance muddies K-means**: K=2 on raw features enriches the pulsar cluster (≈62% pure, most pulsars caught) but never gives the rare class a clean cluster of its own -- and a healthy silhouette hides that.
- **Silhouette is necessary, not sufficient**: it scores geometry, not whether you
  found the science class. Validate against what you know whenever you can.
- **Order matters**: reduce → cluster beats cluster-on-raw-features.
- **Match the reduction to the method**: K-means ↔ PCA (convex blobs), DBSCAN ↔ UMAP (density gaps). The representation decides what the algorithm can find.

## Part 6 — Your turn

You've watched the full pipeline run. Now turn a few knobs yourself. Everything you
need is already in memory: `X`, `y`, `cols`, the scaled features `Xs`, and the PCA
embedding `emb_pca`. A small helper for scoring clusters against the pulsar label:

In [ ]:
def purity_completeness(labels, y, cluster_id):
    """purity      = fraction of this cluster that are real pulsars
       completeness = fraction of all pulsars that landed in this cluster"""
    in_cluster = labels == cluster_id
    purity = y[in_cluster].mean()
    completeness = in_cluster[y == 1].mean()
    return purity, completeness

### Exercise 1 — Does more clusters help?

In the demo we used K=2 on `emb_pca`. Try **K = 2, 3, 4, 5**. For each K, find the
cluster with the highest pulsar fraction and print its purity and completeness
(use `purity_completeness`).

*Hint:* loop over K, fit `KMeans(n_clusters=K, n_init=10, random_state=0)` on
`emb_pca`, then `np.argmax` over the per-cluster pulsar fractions.

In [ ]:
# YOUR CODE HERE


**Think about it:** the K=2 cluster was only ≈62% pure yet still *caught most of
the pulsars*. How can a cluster be impure and complete at the same time?

> *Purity* (precision) and *completeness* (recall) are independent axes. The
> enriched cluster is large enough to scoop up most real pulsars (high
> completeness) but also drags in a comparable pile of noise (purity stuck at
> ~0.62 — roughly 3 pulsars for every 2 impostors). With a rare class (~9%), even
> a well-placed cluster collects a lot of majority contamination. "Caught most of
> them" says nothing about "how many false alarms came along" — you need both
> numbers.

### Exercise 2 — How sensitive is DBSCAN to `eps`?

DBSCAN has no `K`, but `eps` (the neighbourhood radius) decides everything. Sweep a
few values on the PCA embedding and watch the cluster/noise counts move.

We standardise the embedding first so the `eps` values are interpretable (in units
of standard deviations). For each `eps` in **0.2, 0.3, 0.5, 0.8**, fit
`DBSCAN(eps=eps, min_samples=10)` and print the number of clusters and the number
of noise points (label `-1`).

*Hint:* `n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)`.

In [ ]:
from sklearn.preprocessing import StandardScaler
emb = StandardScaler().fit_transform(emb_pca)   # interpretable eps units

# YOUR CODE HERE


### Exercise 3 — Do we even need clustering?

Remember the feature you picked in Setup as the cleanest separator. The strongest
single one is **`prof_kurt`** (excess kurtosis of the integrated profile) — pulsars
have sharp, narrow profiles, so their kurtosis runs high. A simple threshold on it
is already a crude pulsar detector.

For each `thresh` in **0.5, 1.0, 2.0, 4.0**, flag candidates with
`prof_kurt > thresh` and print the purity and completeness of the flagged set
(same definitions as before).

*Hint:* `feat = X[:, cols.index("prof_kurt")]`, then `pred = feat > thresh`.

**Then go further:** `prof_kurt` is just the *easiest* separator. Try the other
features too — loop over all 8 columns and, for a sensible threshold each, print
purity/completeness (or just eyeball the Setup histograms again). How many features
separate the classes well on their own? This tells you how much of the "signal" is
already in single features *before* any clustering — and clustering only earns its
keep when **no single feature** does the job.

In [ ]:
feat = X[:, cols.index("prof_kurt")]   # excess kurtosis of the integrated profile

# YOUR CODE HERE


**Think about it:** a hand-tuned threshold can *rival* the whole clustering
pipeline. So was the clustering pointless?

> No — but mind the catch: we only knew *which* feature and *which* threshold to
> pick because we peeked at the labels. That's supervised information. Clustering's
> job is to find structure **without** labels. On a fresh survey with no labels you
> couldn't build this baseline at all — but you could still run the pipeline and
> *then* check a handful of candidates by hand. The lesson isn't "clustering beats
> a threshold"; it's that the two use different information.